####Installation

In [1]:
#| hide
!pip install -Uqq nixtla

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.7 MB/s eta 0:00:00


####Import TimeGPT client

In [2]:
from nixtla import NixtlaClient

####Client Creation and Verification

In [3]:
nixtla_client = NixtlaClient(
    api_key = 'nixak-bgLEHOWa4jxmToIznMKaXaxtI9EUdim2Z2geIqI6EOjEmmgu2noAvxhWiMv7TL8neMzwSIuj3VIB15d1'
)
nixtla_client.validate_api_key()

True

####Data Collection
  

In [5]:
# Additional imports
from numpy.typing import ArrayLike
import kagglehub
import pandas as pd
from sklearn.preprocessing import StandardScaler
from datetime import datetime, timedelta
from io import StringIO
import requests

def get_weather_data(city: str) -> pd.DataFrame:
    path = kagglehub.dataset_download("gucci1337/weather-of-albania-last-three-years")
    years = [2021, 2022, 2023]
    data_frames = []

    for year in years:
        file_path = f"{path}/data_weather/{city}/{city}{year}.csv"
        df = pd.read_csv(file_path)
        df = df.dropna(subset=['tavg'])  # Remove rows where 'tavg' is NaN
        data_frames.append(df)

    # Concatenate all years' data
    concatenated_data = pd.concat(data_frames, ignore_index=True)
    return concatenated_data

def get_energy_data(year: int) -> pd.DataFrame:
    """
    Downloads daily consumption data for a given year, concatenates all days into a single DataFrame,
    and performs basic preprocessing in-place (e.g., converting the target column to float).

    Parameters:
      year (int): The year for which to retrieve data.

    Returns:
      pd.DataFrame: The combined DataFrame with the target column converted to float.
    """
    all_data = []
    current_date = datetime(year, 1, 1)
    end_date = datetime(year, 12, 31)

    while current_date <= end_date:
        date_str = f"{current_date.day}.{current_date.month}.{current_date.year}"
        url = f"https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t{date_str}"
        print(f"Fetching data for {date_str} from:\n{url}")

        try:
            response = requests.get(url)
            response.raise_for_status()
            daily_df = pd.read_csv(StringIO(response.text), sep=';')
            all_data.append(daily_df)
        except Exception as e:
            print(f"Error fetching data for {date_str}: {e}")
        current_date += timedelta(days=1)

    combined_data = pd.concat(all_data, ignore_index=True)

    # Ensure the target column exists.
    target_col = "Kita Albachten (0005),Strom (S~00000936),[kWh]"
    if target_col in combined_data.columns:
        # Remove commas and convert to numeric.
        combined_data[target_col] = pd.to_numeric(
            combined_data[target_col].str.replace(',', ''), errors='coerce'
        )
    else:
        print(f"Warning: Target column '{target_col}' not found. Available columns: {combined_data.columns.tolist()}")

    return combined_data

def get_country_data(df: pd.DataFrame, country: str, start: int = 200, end: int = 1200) -> pd.DataFrame:
    """
    Returns a DataFrame containing only the data for the given country,
    with NaN values filled using linear interpolation, sorted by 'Date_reported'
    and sliced between the given indices.
    """
    # Filter rows for the given country and sort by 'Date_reported'
    df_country = df[df['Country'] == country].sort_values('Date_reported')
    # Fill NaN values in all numeric columns using linear interpolation
    df_country = df_country.interpolate(method='linear')
    # Slice the DataFrame between start and end indices
    return df_country.iloc[start:end]

def get_healthcare_data(country : str)-> pd.DataFrame:

    # Example usage:
    CSV_FILE_ABSOLUTE_PATH = "/content/WHO-COVID-19-global-daily-data.csv"
    df = pd.read_csv(CSV_FILE_ABSOLUTE_PATH)

    # Get data for Germany (rows 200 to 1200 after sorting by Date_reported).
    df = get_country_data(df, country)
    return df

# weather data
city = "lezhe"
weather_df = get_weather_data(city)

# financial data
finance_df = pd.read_csv("/content/AMZN-stock-price.csv")

# energy data
year = 2024
energy_df = get_energy_data(year)

# healthcare data
country = "Germany"
healthcare_df = get_healthcare_data(country)

Fetching data for 1.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t1.1.2024
Fetching data for 2.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t2.1.2024
Fetching data for 3.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t3.1.2024
Fetching data for 4.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t4.1.2024
Fetching data for 5.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t5.1.2024
Fetching data for 6.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t6.1.2024
Fetching data for 7.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t7.1.2024
Fetching data for 8.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t8.1.2024
Fetching data for 9.1.2024 from:
https://www.eview.de/e1/p3Export.php?fr

<ipython-input-5-8c06c05a10bc>:77: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df_country = df_country.interpolate(method='linear')


In [6]:
finance_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5088 entries, 0 to 5087
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       5088 non-null   object 
 1   Adj Close  5088 non-null   float64
dtypes: float64(1), object(1)
memory usage: 79.6+ KB


#### Forecasting

In [7]:
cols_df_name = [
    ("date", "tavg", weather_df, "weather"),
    ("Date", "Adj Close", finance_df, "finance"),
    ("Datum/Uhrzeit", "Kita Albachten (0005),Strom (S~00000936),[kWh]", energy_df, "energy"),
    ("Date_reported", "New_cases", healthcare_df, "healthcare")
]

scaler = StandardScaler()

final_preds = {}
for time_col, target_col, df, name in cols_df_name:
  df[time_col] = pd.to_datetime(df[time_col])
  date_range = pd.date_range(start=df[time_col].min(), end=df[time_col].max(), freq='D')
  if len(date_range) != len(df):
      print("Warning: Missing dates detected. Reindexing and interpolating missing target values...")
      df = df.set_index(time_col).reindex(date_range).rename_axis(time_col).reset_index()
      df[target_col] = df[target_col].interpolate(method='linear')
  if name not in final_preds.keys():
    final_preds[name] = {}
  df[target_col] = scaler.fit_transform(df[[target_col]])

  df.to_csv(f"{name}.csv")
  print(name)
  for future_context in [32, 64, 128]:
    train_data_stop = int(0.6 * df.shape[0])
    forecast_df = nixtla_client.forecast(
        df=df.iloc[:train_data_stop],
        h=future_context,
        time_col=time_col,
        target_col=target_col,
        finetune_steps=10, # this is what finetunes the model
        finetune_depth=5
    )
    actual_df = df.iloc[train_data_stop:train_data_stop + future_context]
    print(actual_df.shape)
    if future_context not in final_preds[name].keys():
      final_preds[name][future_context] = {
          "forecast" : (time_col, "TimeGPT", forecast_df),
          "actual" : (time_col, target_col, actual_df)
      }
final_preds

/usr/local/lib/python3.11/dist-packages/nixtla/nixtla_client.py:383: UserWarning: `df` contains the following exogenous features: ['tmin', 'tmax', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun'], but `X_df` was not provided and they were not declared in `hist_exog_list`. They will be ignored.
  warnings.warn(


weather


/usr/local/lib/python3.11/dist-packages/nixtla/nixtla_client.py:383: UserWarning: `df` contains the following exogenous features: ['tmin', 'tmax', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun'], but `X_df` was not provided and they were not declared in `hist_exog_list`. They will be ignored.
  warnings.warn(


(32, 11)


/usr/local/lib/python3.11/dist-packages/nixtla/nixtla_client.py:383: UserWarning: `df` contains the following exogenous features: ['tmin', 'tmax', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun'], but `X_df` was not provided and they were not declared in `hist_exog_list`. They will be ignored.
  warnings.warn(


...

/usr/local/lib/python3.11/dist-packages/nixtla/nixtla_client.py:383: UserWarning: `df` contains the following exogenous features: ['Country_code', 'Country', 'WHO_region', 'Cumulative_cases', 'New_deaths', 'Cumulative_deaths'], but `X_df` was not provided and they were not declared in `hist_exog_list`. They will be ignored.
  warnings.warn(


(32, 8)


/usr/local/lib/python3.11/dist-packages/nixtla/nixtla_client.py:383: UserWarning: `df` contains the following exogenous features: ['Country_code', 'Country', 'WHO_region', 'Cumulative_cases', 'New_deaths', 'Cumulative_deaths'], but `X_df` was not provided and they were not declared in `hist_exog_list`. They will be ignored.
  warnings.warn(


(64, 8)
(128, 8)


{'weather': {32: {'forecast': ('date',
    'TimeGPT',
             date   TimeGPT
    0  2022-10-20 -0.148811
    1  2022-10-21 -0.144222
    2  2022-10-22 -0.151632
    3  2022-10-23 -0.152721
    4  2022-10-24 -0.145102
    5  2022-10-25 -0.145961
    6  2022-10-26 -0.138875
    7  2022-10-27 -0.130749
    8  2022-10-28 -0.135804
    9  2022-10-29 -0.151764
    10 2022-10-30 -0.157133
    11 2022-10-31 -0.153024
    12 2022-11-01 -0.144407
    13 2022-11-02 -0.134236
    14 2022-11-03 -0.147908
    15 2022-11-04 -0.150054
    16 2022-11-05 -0.158836
    17 2022-11-06 -0.161468
    18 2022-11-07 -0.161042
    19 2022-11-08 -0.163458
    20 2022-11-09 -0.162095
    21 2022-11-10 -0.168172
    22 2022-11-11 -0.171706
    23 2022-11-12 -0.175736
    24 2022-11-13 -0.176449
    25 2022-11-14 -0.173343
    26 2022-11-15 -0.171347
    27 2022-11-16 -0.170042
    28 2022-11-17 -0.168818
    29 2022-11-18 -0.169157
    30 2022-11-19 -0.171324
    31 2022-11-20 -0.173359),
   'actual': ('date'

####Data Validation and Cleaning

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score, explained_variance_score

# Iterate over each dataset and forecast horizon in final_preds.
for dataset_name, horizons in final_preds.items():
    for horizon, pred_dict in horizons.items():
        # Unpack the tuples.
        # For forecast, the tuple is: (time_col, "TimeGPT", forecast_df)
        # For actual, the tuple is: (time_col, target_col, actual_df)
        f_time_col, _, forecast_df = pred_dict["forecast"]
        a_time_col, target_col, actual_df = pred_dict["actual"]

        # Compute metrics using the target column from the actual tuple.
        mse_val = mean_squared_error(actual_df[target_col], forecast_df["TimeGPT"])
        rmse_val = np.sqrt(mse_val)
        mae_val = mean_absolute_error(actual_df[target_col], forecast_df["TimeGPT"])
        mape_val = mean_absolute_percentage_error(actual_df[target_col], forecast_df["TimeGPT"])
        r2_val = r2_score(actual_df[target_col], forecast_df["TimeGPT"])
        explained_val = explained_variance_score(actual_df[target_col], forecast_df["TimeGPT"])

        print(f"{dataset_name} (Horizon {horizon}) -> RMSE: {rmse_val:.4f}, MSE: {mse_val:.4f}, MAE: {mae_val:.4f}, MAPE: {mape_val:.4f}, R2: {r2_val:.4f}, Explained: {explained_val:.4f}")

        # Prepare the x-axis using the time column.
        try:
            x_forecast = pd.to_datetime(forecast_df[f_time_col])
        except Exception:
            x_forecast = forecast_df.index
        try:
            x_actual = pd.to_datetime(actual_df[a_time_col])
        except Exception:
            x_actual = actual_df.index

        # Create a plot of actual vs. forecast.
        plt.figure(figsize=(10, 5))
        plt.plot(x_actual, actual_df[target_col], label="Actual", marker="o")
        plt.plot(x_forecast, forecast_df["TimeGPT"], label="Forecast", marker="x")
        plt.title(f"{dataset_name.capitalize()} Forecast vs Actual (Horizon: {horizon})")
        plt.xlabel("Date")
        plt.ylabel(target_col)
        plt.legend()
        plt.grid(True)
        plt.tight_layout()

        # Save the plot to a file.
        plot_filename = f"{dataset_name}_{horizon}_forecast_plot.png"
        plt.savefig(plot_filename)
        plt.close()


weather (Horizon 32) -> RMSE: 0.1475, MSE: 0.0218, MAE: 0.1232, MAPE: 2.5538, R2: 0.0776, Explained: 0.1249
weather (Horizon 64) -> RMSE: 0.4730, MSE: 0.2237, MAE: 0.3540, MAPE: 1.6195, R2: -0.4297, Explained: 0.0524
weather (Horizon 128) -> RMSE: 0.7623, MSE: 0.5811, MAE: 0.6329, MAPE: 1.2265, R2: -1.4559, Explained: 0.0305
finance (Horizon 32) -> RMSE: 0.0200, MSE: 0.0004, MAE: 0.0176, MAPE: 0.0456, R2: -0.0441, Explained: 0.0553
finance (Horizon 64) -> RMSE: 0.0161, MSE: 0.0003, MAE: 0.0136, MAPE: 0.0349, R2: -0.1016, Explained: 0.0470
finance (Horizon 128) -> RMSE: 0.0665, MSE: 0.0044, MAE: 0.0419, MAPE: 0.1511, R2: -0.4573, Explained: 0.0005
energy (Horizon 32) -> RMSE: 1.7025, MSE: 2.8985, MAE: 1.3732, MAPE: 4.5324, R2: -1.6417, Explained: -0.0855
energy (Horizon 64) -> RMSE: 1.7892, MSE: 3.2012, MAE: 1.5459, MAPE: 4.5897, R2: -2.8053, Explained: -0.0854
energy (Horizon 128) -> RMSE: 1.8727, MSE: 3.5068, MAE: 1.6614, MAPE: 5.3363, R2: -2.9447, Explained: -0.0467
healthcare (Horiz